# M-ViSER - Speech Emotion Recognition

**Repo**: https://github.com/Huu2412/M-ViSER

---
### Training Modes
| Stage | Mo ta | Lenh |
|---|---|---|
| **0** | End-to-End (Student + Teacher cung luc) | `--stage 0` |
| **1** | Chi train Teacher (Audio + Clean Text) | `--stage 1` |
| **2** | Student Distillation tu Teacher da freeze | `--stage 2 --teacher_ckpt ...` |

> **QUAN TRONG**: Chay tung cell theo thu tu 1 -> 2 -> 3 -> 4 -> 5x


In [ ]:
# ============================================================
# CELL 1: Clone repo (luon lay code moi nhat)
# ============================================================
import os

REPO_URL = 'https://github.com/Huu2412/M-ViSER.git'
REPO_DIR = '/kaggle/working/M-ViSER'

os.system(f'rm -rf {REPO_DIR}')
os.system(f'git clone {REPO_URL} {REPO_DIR}')

print('=== Latest 3 commits ===')
os.system(f'git -C {REPO_DIR} log --oneline -3')


In [ ]:
# ============================================================
# CELL 2: Cai dat dependencies tu requirements.txt
# ============================================================
import os
os.system(f'pip install -r {REPO_DIR}/requirements.txt')
os.system('pip uninstall -y torchcodec')


In [ ]:
# ============================================================
# CELL 3: Kiem tra GPU chi tiet
# (Neu Cell 2 bao RESTART KERNEL -> restart truoc khi chay cell nay)
# ============================================================
import torch, os

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {props.total_memory/1e9:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'Compute Cap.    : sm_{props.major}{props.minor}')

    try:
        _x = torch.tensor([1.0, float('nan')]).cuda()
        torch.isfinite(_x)
        del _x
        print('CUDA kernel test: PASSED')
    except Exception as e:
        print(f'CUDA kernel test: FAILED -> {e}')
        print('Quay lai Cell 2, sau do Restart Kernel!')
else:
    print('WARNING: Khong co GPU! Hay bat GPU trong Settings.')

# Kiem tra version >= 2.6
from packaging.version import Version
tv = Version(torch.__version__.split('+')[0])
if tv < Version('2.6.0'):
    print(f'WARNING: torch {torch.__version__} < 2.6.0 -> transformers se tu choi!')
    print('Hay chay lai Cell 2 de upgrade.')
else:
    print(f'Version check   : OK ({torch.__version__} >= 2.6.0)')

os.system('df -h /kaggle/working')


In [ ]:
# ============================================================
# CELL 4: Smoke Test (forward + backward pass)
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'Working dir: {os.getcwd()}')
# ret = os.system('CUDA_LAUNCH_BLOCKING=1 python smoke_test.py')
# print('\nSmoke test: PASSED' if ret == 0 else '\nSmoke test: FAILED')


In [ ]:
# ============================================================
# CELL 5A: Train -- End-to-End (Stage 0)
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
# os.system('python train.py --config config/config.yaml --stage 0')


In [ ]:
# ============================================================
# CELL 5B: Train -- Stage 1: Teacher Only
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
os.system('python train.py --config config/config.yaml --stage 1')


In [ ]:
# ============================================================
# CELL 5C: Train -- Stage 2: Student Distillation
#   Can chay Cell 5B truoc!
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

TEACHER_CKPT = f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt'
if not os.path.exists(TEACHER_CKPT):
    print(f'Teacher checkpoint khong tim thay: {TEACHER_CKPT}')
    print('  --> Hay chay Cell 5B (Stage 1) truoc!')
else:
    print(f'Teacher checkpoint: {TEACHER_CKPT}')
    # os.system(
        f'python train.py --config config/config.yaml '
# f'--stage 2 --teacher_ckpt {TEACHER_CKPT}'
    # )


In [ ]:
# ============================================================
# CELL 6: 5-Fold Cross-Validation
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
FOLDS = '1 2 3 4 5'
# os.system(f'python run_5fold.py --config config/config.yaml --folds {FOLDS}')


In [ ]:
# ============================================================
# CELL 7: Evaluate tren test set
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

BEST_CKPT = f'{REPO_DIR}/checkpoints/best_model.pt'
for c in [
    f'{REPO_DIR}/checkpoints/stage2_student/best_model.pt',
    f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt',
]:
    if not os.path.exists(BEST_CKPT) and os.path.exists(c):
        BEST_CKPT = c

print(f'Evaluating: {BEST_CKPT}')
os.system(f'python evaluate.py --config config/config.yaml --checkpoint {BEST_CKPT}')


In [ ]:
# ============================================================
# CELL 8: Nen va export checkpoint
# ============================================================
import os, shutil
from datetime import datetime

OUTPUT_DIR = '/kaggle/working'
CKPT_DIR   = f'{REPO_DIR}/checkpoints'
ts         = datetime.now().strftime('%Y%m%d_%H%M')
zip_base   = f'{OUTPUT_DIR}/mvisar_ckpt_{ts}'

if os.path.exists(CKPT_DIR):
    shutil.make_archive(zip_base, 'zip', CKPT_DIR)
    zip_file = zip_base + '.zip'
    size_mb  = os.path.getsize(zip_file) / 1e6
    print(f'Done: {zip_file} ({size_mb:.1f} MB)')
    print('Tai ve: Kaggle > Output tab')
else:
    print(f'Khong tim thay: {CKPT_DIR}')

print('\n=== Kaggle Output ===')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  {fname}: {os.path.getsize(fpath)/1e6:.1f} MB')
